# headswap — expression chain

**T4 head swap → LivePortrait expression transfer.**

**Cell 1** (setup + load models) → **Cell 2** (upload a pair) → **Cell 3** (run, ~21s every time).

Cell 1 is the slow one: it installs anything missing, loads both engines into this kernel, and runs one throwaway pass to pay CUDA warm-up. It warms on a pair bundled in the repo, so you don't need to upload first.

> **On a brand-new runtime Cell 1 restarts the kernel after installing.** That is expected — just **run Cell 1 again**; the second time it skips the install and loads the models.

After that, Cell 3 costs the warm number on **every** run. New pair → re-run Cell 2, then Cell 3. Only re-run Cell 1 if the runtime restarts.

Settings fixed to the arm that worked: LivePortrait runs **after** the swap (before it, T4 regenerates the head and the expression is lost), `driving_multiplier=0.8`, `animation_region=lip`, `face_refine` skipped on bust shots.


In [ ]:
#@title Cell 1 - Setup AND load models  (re-run once after the auto-restart)
from pathlib import Path
import subprocess, shutil, os, signal, sys

assert Path("/content").exists(), "Open this notebook in Google Colab."
import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then re-run.")
print(f"GPU {torch.cuda.get_device_name(0)}")

REPO = Path("/content/headswap_V2")
BRANCH = "simple-full-body-head-swap"
if not REPO.exists():
    subprocess.run(["git", "clone",
                    "https://github.com/malihashar/headswap_V2.git", str(REPO)],
                   check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH,
                f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "-C", str(REPO), "reset", "--hard", f"origin/{BRANCH}"],
               check=True)
print("HEAD:", subprocess.getoutput(f"git -C {REPO} rev-parse --short HEAD"))
os.chdir(REPO)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)

# --- install (fresh runtime only) -----------------------------------------
# This branch ENDS the process: newly installed custom nodes and pinned
# numpy are not visible to an already-running kernel, so models cannot be
# loaded in the same execution. Re-running the cell lands in the else-branch
# and proceeds straight to loading.
if not Path("/content/ComfyUI/server.py").exists():
    print("-> Fresh runtime: installing ComfyUI + Krea2 weights (~20GB, slow)")
    shutil.rmtree("/content/ComfyUI", ignore_errors=True)
    r = subprocess.run(["bash", "scripts/setup_colab.sh", "--no-drive", "--krea2"],
                       check=False, capture_output=True, text=True)
    print("setup exit:", r.returncode); print(r.stdout[-1200:])
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise SystemExit("setup_colab.sh failed")
    subprocess.run(["pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
                    "--no-deps", "numpy==2.4.6"], check=True)
    print("\n✓ Installed. Restarting kernel (expected).")
    print("   >>> RUN THIS CELL AGAIN to load the models. <<<")
    os.kill(os.getpid(), signal.SIGKILL)

print("ComfyUI present - skipping install.")

# --- LivePortrait ----------------------------------------------------------
LP = Path("/content/LivePortrait")
if not (LP / "inference.py").exists():
    print("-> installing LivePortrait ...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/KwaiVGI/LivePortrait", str(LP)], check=True)
    subprocess.run(["pip", "install", "-q", "--no-cache-dir", "tyro", "imageio",
                    "imageio-ffmpeg", "rich", "pykalman", "ffmpeg-python"],
                   check=False)
if not (LP / "pretrained_weights" / "liveportrait").exists():
    print("-> fetching LivePortrait weights (~660MB) ...")
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="KwaiVGI/LivePortrait",
                      local_dir=str(LP / "pretrained_weights"),
                      ignore_patterns=["*animal*"])

# --- simple-lama (headwear erase) -----------------------------------------
# Only setup_colab.sh installs this, and that is skipped whenever ComfyUI is
# already present -- so a reconnected runtime silently lacks it and
# erase_headwear() falls back, returning the image unchanged with the hat
# still on it.
#
# It must be installed with the pin repair: simple-lama-inpainting pulls
# pillow 9.5 and numpy 1.26 as transitive deps, which breaks rembg and
# restore_background with NO error -- they just start falling back. That is a
# documented trap in this repo, so the pins are restored immediately after.
try:
    import simple_lama_inpainting  # noqa: F401
    print("simple-lama present.")
except ImportError:
    print("-> installing simple-lama-inpainting (+ repairing its pin damage) ...")
    subprocess.run(["pip", "install", "-q", "simple-lama-inpainting"], check=False)
    subprocess.run(["pip", "install", "-q", "--force-reinstall", "--no-deps",
                    "pillow==11.3.0"], check=False)
    subprocess.run(["pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
                    "--no-deps", "numpy==2.4.6"], check=False)
    try:
        import simple_lama_inpainting  # noqa: F401
        print("   simple-lama OK")
    except ImportError as _e:
        print(f"   WARN: still unavailable ({_e}); headwear erase will skip")

# --- load models into THIS kernel -----------------------------------------
import importlib.util
_s = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
_ce = importlib.util.module_from_spec(_s); _s.loader.exec_module(_ce)
PATHS = _ce.apply_env(_ce.default_paths(use_drive=False))
_ce.ensure_import_path(REPO)
_comfy = str(PATHS.get("comfyui", "/content/ComfyUI"))
if _comfy not in sys.path:
    sys.path.insert(0, _comfy)
for _m in [m for m in sys.modules if m.startswith("headswap")]:
    del sys.modules[_m]

from headswap.chain import warmup

# Warm on a pair bundled in the repo, so no upload is needed first.
WARM_BODY = REPO / "data" / "eval" / "bodies" / "pair_000.png"
WARM_FACE = REPO / "data" / "eval" / "faces" / "pair_000.png"
print("\nLoading models + one throwaway pass (the slow part) ...\n")
warmup(WARM_BODY, WARM_FACE, lp_dir=LP)
print("\n✓ Models resident in this kernel. Run Cell 2, then Cell 3 (~21s).")


In [ ]:
#@title Cell 2 - Upload a pair
# BODY  = the photo you keep (pose, clothing, background) AND whose
#         expression you want on the final face.
# FACE  = the donor whose identity is transferred in.
import os, uuid
from pathlib import Path
from PIL import Image
from IPython.display import display
from google.colab import files

REPO = Path("/content/headswap_V2")
PAIR = REPO / "data" / "custom" / "chain_pair"
PAIR.mkdir(parents=True, exist_ok=True)
for old in PAIR.glob("*"):
    old.unlink()

def grab(role, what):
    print(f"\n=== Upload the {role.upper()} image ({what}) ===")
    up = files.upload()
    if not up:
        raise SystemExit(f"No {role} uploaded - re-run this cell.")
    name = next(iter(up))
    Image.open(name).convert("RGB").save(PAIR / f"{role}.png")
    os.remove(name)
    im = Image.open(PAIR / f"{role}.png")
    print(f"  saved {role}: {im.size[0]}x{im.size[1]}")
    return im

body = grab("body", "TARGET: pose/clothes/background + the expression you want")
face = grab("face", "DONOR: the identity to transfer in")
display(body); display(face)
print("\n✓ Ready. Run Cell 3.")


In [ ]:
#@title Cell 3 - RUN (re-runnable; this is the real per-request time)
import time
from pathlib import Path
from IPython.display import Image as IPImage, display, Markdown

from headswap.chain import run_chain

REPO = Path("/content/headswap_V2")
PAIR = REPO / "data" / "custom" / "chain_pair"
OUT = REPO / "results" / "chain_run"

t0 = time.perf_counter()
r = run_chain(PAIR / "body.png", PAIR / "face.png",
              out_dir=OUT, lp_dir="/content/LivePortrait")
wall = time.perf_counter() - t0

print("\n" + "=" * 52)
print(f"  T4 swap        {r['swap_s']:6.1f}s")
print(f"  LivePortrait   {r['lp_s']:6.1f}s   (applied={r['lp_applied']})")
print(f"  TOTAL          {r['total_s']:6.1f}s   (cell wall {wall:.1f}s)")
print("=" * 52)
if not r["was_warm"]:
    print("  NOTE: models were not warmed - run Cell 3 first for a real number.")

display(Markdown("### T4 swap only"))
display(IPImage(filename=str(r["swap_only"])))
display(Markdown("### Final (after LivePortrait)"))
display(IPImage(filename=str(r["final"])))


In [ ]:
#@title Cell 4 - Grounding sweep (carrier 3)  -- run after Cell 2
# Krea2EditGroundedEncode runs a VLM over the TARGET at `grounding_px` before
# the prompt is interpreted. It is the only scene-conditioning path never
# swept: ref_boost_a scales a different one (0.3 left the cap, wrecked the
# polo) and denoise governs a third (cannot drop it without losing clothes).
#
# Lower grounding = a weaker commitment to "this person is wearing a cap"
# while the prompt still says the cap is gone. Same question for the robe.
#
# ~90s per arm, in this kernel (no reload). 4 arms ~= 6 min.
import time
from pathlib import Path
from IPython.display import Image as IPImage, display, Markdown

from headswap.chain import sweep_grounding

REPO = Path("/content/headswap_V2")
PAIR = REPO / "data" / "custom" / "chain_pair"

t0 = time.perf_counter()
s = sweep_grounding(PAIR / "body.png", PAIR / "face.png",
                    out_dir=REPO / "results" / "grounding_sweep",
                    grounding_px=(768, 512, 384, 256))
print(f"\nsweep wall {time.perf_counter() - t0:.0f}s")

# Check the knob MOVED before reading the pictures. Two arms sharing an
# effective value means the setting never reached the node, and the images
# are not evidence of anything.
eff = [a["effective_px"] for a in s["arms"]]
print("\neffective grounding_px per arm:", eff)
if len(set(eff)) != len(eff) or None in eff:
    print("  !! the knob did NOT move on every arm - do not read the montage")

display(Markdown("### grounding sweep (target, donor, then each arm)"))
display(IPImage(filename=s["montage"]))
for a in s["arms"]:
    display(Markdown(f"**grounding_px = {a['effective_px']}**  ({a['seconds']}s)"))
    display(IPImage(filename=a["path"]))
